In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ══════════════════════════════════════════════
#  1. 2D CONVOLUTION FROM SCRATCH
# ══════════════════════════════════════════════

class Conv2dScratch(nn.Module):
    """
    2D Convolution — from scratch using loops.

    How it works (for ONE output pixel):
        out[i,j] = Σ_c Σ_kh Σ_kw  input[c, i*s+kh, j*s+kw] * kernel[c, kh, kw]  + bias

    Parameters:
        in_channels:  input depth  (e.g. 3 for RGB)
        out_channels: number of filters (each produces one output channel)
        kernel_size:  filter height & width
        stride:       step size
        padding:      zero-padding added to input edges
    """

    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int, stride: int = 1, padding: int = 0):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.padding      = padding

        # Learnable parameters (same shape as nn.Conv2d)
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.1
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x shape: (batch, in_channels, H, W)
        output:  (batch, out_channels, H_out, W_out)
        """
        # Pad input if needed
        if self.padding > 0:
            x = F.pad(x, [self.padding] * 4)   # pad all sides equally

        B, C, H, W = x.shape
        K = self.kernel_size
        S = self.stride

        # Output spatial dimensions
        H_out = (H - K) // S + 1
        W_out = (W - K) // S + 1

        # Allocate output
        out = torch.zeros(B, self.out_channels, H_out, W_out, device=x.device)

        # Slide the kernel across the input
        for i in range(H_out):
            for j in range(W_out):
                # Extract the patch under the kernel
                h_start = i * S
                w_start = j * S
                patch = x[:, :, h_start:h_start+K, w_start:w_start+K]
                # patch shape: (B, in_channels, K, K)

                # Convolve: multiply each filter with the patch and sum
                for f in range(self.out_channels):
                    out[:, f, i, j] = (patch * self.weight[f]).sum(dim=(1, 2, 3)) + self.bias[f]

        return out


# ══════════════════════════════════════════════
#  2. FAST CONV2D (im2col approach — no inner loops)
# ══════════════════════════════════════════════

class Conv2dFast(nn.Module):
    """
    2D Convolution using unfold (im2col) — vectorised, no loops.
    Same result as Conv2dScratch, but much faster.

    Trick: unfold extracts all patches as columns → one big matmul.
    """

    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int, stride: int = 1, padding: int = 0):
        super().__init__()
        self.out_channels = out_channels
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.padding      = padding

        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.1
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        K, S, P = self.kernel_size, self.stride, self.padding

        # Unfold: extract all K×K patches → (B, C*K*K, n_patches)
        x_unfold = F.unfold(x, kernel_size=K, stride=S, padding=P)

        # Reshape weight: (out_channels, C*K*K)
        w_flat = self.weight.view(self.out_channels, -1)

        # Matrix multiply: (out_channels, C*K*K) @ (C*K*K, n_patches) → (out_channels, n_patches)
        out = w_flat @ x_unfold + self.bias.unsqueeze(1)
        # out shape: (B, out_channels, n_patches)

        # Reshape back to spatial
        H_out = (H + 2 * P - K) // S + 1
        W_out = (W + 2 * P - K) // S + 1
        return out.view(B, self.out_channels, H_out, W_out)


# ══════════════════════════════════════════════
#  3. MAX POOLING FROM SCRATCH
# ══════════════════════════════════════════════

class MaxPool2dScratch(nn.Module):
    """
    Max Pooling — takes the maximum value in each kernel-sized window.

    No learnable parameters. Just downsamples by keeping the strongest
    activation in each local region.

        out[i,j] = max( input[i*s : i*s+k, j*s : j*s+k] )
    """

    def __init__(self, kernel_size: int, stride: int = None):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size  # default: non-overlapping

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x shape: (batch, channels, H, W)
        output:  (batch, channels, H_out, W_out)
        """
        B, C, H, W = x.shape
        K = self.kernel_size
        S = self.stride

        H_out = (H - K) // S + 1
        W_out = (W - K) // S + 1

        out = torch.zeros(B, C, H_out, W_out, device=x.device)

        for i in range(H_out):
            for j in range(W_out):
                h_start = i * S
                w_start = j * S
                patch = x[:, :, h_start:h_start+K, w_start:w_start+K]
                out[:, :, i, j] = patch.reshape(B, C, -1).max(dim=2).values

        return out


# ══════════════════════════════════════════════
#  4. EXAMPLE USAGE
# ══════════════════════════════════════════════

if __name__ == "__main__":

    print("=" * 60)
    print("  2D Convolution & Max Pooling from Scratch")
    print("=" * 60)

    # ──────────────────────────────────────
    # Demo 1: See convolution step by step
    # ──────────────────────────────────────
    print("\n── Demo 1: Convolution step by step ──\n")

    # 1 image, 1 channel, 5×5
    x = torch.tensor([[
        [1, 2, 0, 1, 3],
        [0, 1, 2, 3, 1],
        [1, 0, 1, 0, 2],
        [2, 1, 0, 1, 0],
        [0, 3, 2, 1, 1],
    ]], dtype=torch.float32).unsqueeze(0)   # (1, 1, 5, 5)

    conv = Conv2dScratch(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0)
    # Set a simple edge-detection kernel
    conv.weight.data = torch.tensor([[
        [[-1, -1, -1],
         [-1,  8, -1],
         [-1, -1, -1]]
    ]], dtype=torch.float32)
    conv.bias.data = torch.zeros(1)

    out = conv(x)

    print(f"  Input (5×5):")
    for row in x[0, 0].tolist():
        print(f"    {row}")

    print(f"\n  Kernel (3×3 edge detector):")
    for row in conv.weight.data[0, 0].tolist():
        print(f"    {[int(v) for v in row]}")

    print(f"\n  Output (3×3):")
    for row in out[0, 0].tolist():
        print(f"    {[f'{v:.0f}' for v in row]}")

    print(f"\n  Shapes: {list(x.shape)} → conv3×3 → {list(out.shape)}")

    # ──────────────────────────────────────
    # Demo 2: Max Pooling step by step
    # ──────────────────────────────────────
    print(f"\n── Demo 2: Max Pooling step by step ──\n")

    x_pool = torch.tensor([[
        [1, 3, 2, 4],
        [5, 6, 1, 2],
        [0, 2, 8, 3],
        [7, 1, 4, 5],
    ]], dtype=torch.float32).unsqueeze(0)   # (1, 1, 4, 4)

    pool = MaxPool2dScratch(kernel_size=2, stride=2)
    out_pool = pool(x_pool)

    print(f"  Input (4×4):")
    for row in x_pool[0, 0].tolist():
        print(f"    {[int(v) for v in row]}")

    print(f"\n  MaxPool2d(kernel=2, stride=2):")
    print(f"    Top-left  2×2: max(1,3,5,6) = 6")
    print(f"    Top-right 2×2: max(2,4,1,2) = 4")
    print(f"    Bot-left  2×2: max(0,2,7,1) = 7")
    print(f"    Bot-right 2×2: max(8,3,4,5) = 8")

    print(f"\n  Output (2×2):")
    for row in out_pool[0, 0].tolist():
        print(f"    {[int(v) for v in row]}")

    print(f"\n  Shapes: {list(x_pool.shape)} → pool2×2 → {list(out_pool.shape)}")

    # ──────────────────────────────────────
    # Demo 3: Verify against PyTorch
    # ──────────────────────────────────────
    print(f"\n── Demo 3: Verify against PyTorch ──\n")

    torch.manual_seed(42)
    x_test = torch.randn(2, 3, 8, 8)   # batch=2, RGB, 8×8

    # Conv2d
    conv_ours = Conv2dScratch(3, 4, kernel_size=3, stride=1, padding=1)
    conv_pt   = nn.Conv2d(3, 4, kernel_size=3, stride=1, padding=1)

    # Copy same weights
    conv_pt.weight.data = conv_ours.weight.data.clone()
    conv_pt.bias.data   = conv_ours.bias.data.clone()

    out_ours = conv_ours(x_test)
    out_pt   = conv_pt(x_test)
    conv_match = torch.allclose(out_ours, out_pt, atol=1e-5)

    print(f"  Conv2d(3→4, k=3, s=1, p=1)")
    print(f"    Input:     {list(x_test.shape)}")
    print(f"    Our shape: {list(out_ours.shape)}")
    print(f"    PT  shape: {list(out_pt.shape)}")
    print(f"    Match:     {conv_match} ✓" if conv_match else f"    Match: {conv_match} ✗")

    # Conv2d Fast (im2col)
    conv_fast = Conv2dFast(3, 4, kernel_size=3, stride=1, padding=1)
    conv_fast.weight.data = conv_ours.weight.data.clone()
    conv_fast.bias.data   = conv_ours.bias.data.clone()
    out_fast = conv_fast(x_test)
    fast_match = torch.allclose(out_fast, out_pt, atol=1e-5)
    print(f"\n  Conv2dFast (im2col)")
    print(f"    Match PT:  {fast_match} ✓" if fast_match else f"    Match PT: {fast_match} ✗")

    # MaxPool2d
    pool_ours = MaxPool2dScratch(kernel_size=2, stride=2)
    pool_pt   = nn.MaxPool2d(kernel_size=2, stride=2)

    pool_out_ours = pool_ours(out_pt)
    pool_out_pt   = pool_pt(out_pt)
    pool_match = torch.allclose(pool_out_ours, pool_out_pt, atol=1e-5)

    print(f"\n  MaxPool2d(k=2, s=2)")
    print(f"    Input:     {list(out_pt.shape)}")
    print(f"    Our shape: {list(pool_out_ours.shape)}")
    print(f"    PT  shape: {list(pool_out_pt.shape)}")
    print(f"    Match:     {pool_match} ✓" if pool_match else f"    Match: {pool_match} ✗")

    # ──────────────────────────────────────
    # Demo 4: Build a mini CNN
    # ──────────────────────────────────────
    print(f"\n── Demo 4: Mini CNN with our layers ──\n")

    class MiniCNN(nn.Module):
        """Small CNN using our from-scratch layers."""
        def __init__(self):
            super().__init__()
            self.conv1 = Conv2dFast(1, 8, kernel_size=3, padding=1)
            self.pool1 = MaxPool2dScratch(kernel_size=2)
            self.conv2 = Conv2dFast(8, 16, kernel_size=3, padding=1)
            self.pool2 = MaxPool2dScratch(kernel_size=2)
            self.fc    = nn.Linear(16 * 7 * 7, 10)   # for 28×28 input

        def forward(self, x):
            x = self.pool1(torch.relu(self.conv1(x)))   # 28→14
            x = self.pool2(torch.relu(self.conv2(x)))   # 14→7
            x = x.view(x.size(0), -1)                   # flatten
            return self.fc(x)

    cnn = MiniCNN()
    fake_mnist = torch.randn(4, 1, 28, 28)   # batch of 4 "MNIST" images
    logits = cnn(fake_mnist)

    print(f"  Architecture:")
    print(f"    Conv(1→8, k=3, p=1) → ReLU → MaxPool(2)")
    print(f"    Conv(8→16, k=3, p=1) → ReLU → MaxPool(2)")
    print(f"    Flatten → Linear(784→10)")
    print(f"\n  Input:  {list(fake_mnist.shape)}")
    print(f"  Output: {list(logits.shape)}  (10 class logits)")

    n_params = sum(p.numel() for p in cnn.parameters())
    print(f"  Params: {n_params:,}")

    # Quick training step
    target = torch.randint(0, 10, (4,))
    loss = nn.CrossEntropyLoss()(logits, target)
    loss.backward()
    print(f"  Loss:   {loss.item():.4f}  (random init, untrained)")
    print(f"  ✓ Backward pass works — gradients flow through our custom layers!")

    # ── Summary ──
    print(f"""
══════════════════════════════════════════════════════════════
  HOW CONVOLUTION WORKS

  Input (C_in, H, W)  +  Filter (C_in, K, K)  =  one output pixel

  1. Place the K×K filter at position (i, j)
  2. Element-wise multiply filter × input patch
  3. Sum everything → one number
  4. Slide to next position (by stride)
  5. Repeat for each of the out_channels filters

  Output size: (H + 2P - K) / S + 1

  HOW MAX POOLING WORKS

  1. Slide a K×K window across the input
  2. Take the MAX value in each window
  3. No learnable parameters — just downsampling
  4. Reduces spatial size, keeps strongest features
══════════════════════════════════════════════════════════════
""")

  2D Convolution & Max Pooling from Scratch

── Demo 1: Convolution step by step ──

  Input (5×5):
    [1.0, 2.0, 0.0, 1.0, 3.0]
    [0.0, 1.0, 2.0, 3.0, 1.0]
    [1.0, 0.0, 1.0, 0.0, 2.0]
    [2.0, 1.0, 0.0, 1.0, 0.0]
    [0.0, 3.0, 2.0, 1.0, 1.0]

  Kernel (3×3 edge detector):
    [-1, -1, -1]
    [-1, 8, -1]
    [-1, -1, -1]

  Output (3×3):
    ['1', '8', '14']
    ['-8', '0', '-10']
    ['-1', '-9', '1']

  Shapes: [1, 1, 5, 5] → conv3×3 → [1, 1, 3, 3]

── Demo 2: Max Pooling step by step ──

  Input (4×4):
    [1, 3, 2, 4]
    [5, 6, 1, 2]
    [0, 2, 8, 3]
    [7, 1, 4, 5]

  MaxPool2d(kernel=2, stride=2):
    Top-left  2×2: max(1,3,5,6) = 6
    Top-right 2×2: max(2,4,1,2) = 4
    Bot-left  2×2: max(0,2,7,1) = 7
    Bot-right 2×2: max(8,3,4,5) = 8

  Output (2×2):
    [6, 4]
    [7, 8]

  Shapes: [1, 1, 4, 4] → pool2×2 → [1, 1, 2, 2]

── Demo 3: Verify against PyTorch ──

  Conv2d(3→4, k=3, s=1, p=1)
    Input:     [2, 3, 8, 8]
    Our shape: [2, 4, 8, 8]
    PT  shape: [2, 4, 8